# 10 Image-Related Pretrained Models Using Hugging Face Pipelines

This notebook demonstrates ten image-related pretrained models using the Hugging Face `pipeline()` API.

## Examples

1. ViT image classification  
2. ResNet image classification  
3. CLIP zero-shot image classification  
4. DETR object detection  
5. Mask2Former image segmentation  
6. Depth Anything depth estimation  
7. BLIP image captioning  
8. ViLT visual question answering  
9. OWL-ViT zero-shot object detection  
10. DINOv2 image feature extraction and similarity  

The first run downloads model files from the Hugging Face Hub.


## Install required libraries

In [ ]:
%pip install -q -U transformers torch torchvision pillow requests matplotlib accelerate timm scipy

## Imports and common helper functions

In [ ]:
import io
import requests
import numpy as np
import torch
import matplotlib.pyplot as plt

from PIL import Image, ImageDraw
from transformers import pipeline

DEVICE = 0 if torch.cuda.is_available() else -1

print("GPU available:", torch.cuda.is_available())
print("Pipeline device:", "GPU" if DEVICE == 0 else "CPU")


In [ ]:
def load_image(source):
    if isinstance(source, Image.Image):
        return source.convert("RGB")

    if isinstance(source, str) and source.startswith(("http://", "https://")):
        response = requests.get(source, timeout=30)
        response.raise_for_status()
        return Image.open(io.BytesIO(response.content)).convert("RGB")

    return Image.open(source).convert("RGB")


def show_image(image, title=None, figsize=(8, 6), cmap=None):
    plt.figure(figsize=figsize)
    plt.imshow(image, cmap=cmap)
    plt.axis("off")

    if title:
        plt.title(title)

    plt.show()


def print_predictions(results):
    for index, result in enumerate(results, start=1):
        label = result.get("label", result.get("generated_text", ""))
        score = result.get("score")

        if score is None:
            print(f"{index}. {label}")
        else:
            print(f"{index}. {label}: {score:.4f}")


def draw_boxes(image, detections, threshold=0.3):
    output = image.copy()
    draw = ImageDraw.Draw(output)

    for item in detections:
        if item["score"] < threshold:
            continue

        box = item["box"]
        xmin = int(box["xmin"])
        ymin = int(box["ymin"])
        xmax = int(box["xmax"])
        ymax = int(box["ymax"])

        draw.rectangle(
            (xmin, ymin, xmax, ymax),
            outline="red",
            width=4
        )

        draw.text(
            (xmin + 4, ymin + 4),
            f"{item['label']}: {item['score']:.2f}",
            fill="red"
        )

    return output


## Load sample images

In [ ]:
IMAGE_URL_1 = (
    "https://huggingface.co/datasets/"
    "huggingface/documentation-images/"
    "resolve/main/coco_sample.png"
)

IMAGE_URL_2 = (
    "https://huggingface.co/datasets/"
    "Narsil/image_dummy/"
    "resolve/main/parrots.png"
)

image1 = load_image(IMAGE_URL_1)
image2 = load_image(IMAGE_URL_2)

show_image(image1, "Sample image 1")
show_image(image2, "Sample image 2")


# Example 1: ViT Image Classification

**Pipeline task:** `image-classification`  
**Pretrained model:** `google/vit-base-patch16-224`


In [ ]:
vit_classifier = pipeline(
    task="image-classification",
    model="google/vit-base-patch16-224",
    device=DEVICE
)

vit_results = vit_classifier(image2, top_k=5)
print_predictions(vit_results)


# Example 2: ResNet Image Classification

**Pipeline task:** `image-classification`  
**Pretrained model:** `microsoft/resnet-50`


In [ ]:
resnet_classifier = pipeline(
    task="image-classification",
    model="microsoft/resnet-50",
    device=DEVICE
)

resnet_results = resnet_classifier(image2, top_k=5)
print_predictions(resnet_results)


# Example 3: CLIP Zero-Shot Image Classification

**Pipeline task:** `zero-shot-image-classification`  
**Pretrained model:** `openai/clip-vit-base-patch32`


In [ ]:
clip_classifier = pipeline(
    task="zero-shot-image-classification",
    model="openai/clip-vit-base-patch32",
    device=DEVICE
)

candidate_labels = [
    "birds",
    "dogs",
    "cars",
    "people",
    "food"
]

clip_results = clip_classifier(
    image2,
    candidate_labels=candidate_labels
)

print_predictions(clip_results)


# Example 4: DETR Object Detection

**Pipeline task:** `object-detection`  
**Pretrained model:** `facebook/detr-resnet-50`


In [ ]:
object_detector = pipeline(
    task="object-detection",
    model="facebook/detr-resnet-50",
    device=DEVICE
)

detection_results = object_detector(
    image1,
    threshold=0.5
)

for result in detection_results:
    print(result)

detected_image = draw_boxes(
    image1,
    detection_results,
    threshold=0.5
)

show_image(detected_image, "DETR object detection")


# Example 5: Image Segmentation

**Pipeline task:** `image-segmentation`  
**Pretrained model:** `facebook/mask2former-swin-small-coco-panoptic`


In [ ]:
segmenter = pipeline(
    task="image-segmentation",
    model="facebook/mask2former-swin-small-coco-panoptic",
    device=DEVICE
)

segmentation_results = segmenter(image1)

print("Number of segments:", len(segmentation_results))

for result in segmentation_results[:5]:
    print(
        "Label:", result.get("label"),
        "| Score:", result.get("score")
    )

for result in segmentation_results[:4]:
    show_image(
        result["mask"],
        title=result.get("label", "Segment"),
        figsize=(6, 4),
        cmap="gray"
    )


# Example 6: Depth Estimation

**Pipeline task:** `depth-estimation`  
**Pretrained model:** `depth-anything/Depth-Anything-V2-Small-hf`


In [ ]:
depth_estimator = pipeline(
    task="depth-estimation",
    model="depth-anything/Depth-Anything-V2-Small-hf",
    device=DEVICE
)

depth_result = depth_estimator(image1)

print("Returned keys:", depth_result.keys())

show_image(image1, "Original image")
show_image(
    depth_result["depth"],
    "Estimated depth map",
    cmap="inferno"
)


# Example 7: BLIP Image Captioning

**Pipeline task:** `image-to-text`  
**Pretrained model:** `Salesforce/blip-image-captioning-base`


In [ ]:
captioner = pipeline(
    task="image-to-text",
    model="Salesforce/blip-image-captioning-base",
    device=DEVICE
)

caption_results = captioner(
    image1,
    max_new_tokens=40
)

print(caption_results)
print("Caption:", caption_results[0]["generated_text"])


# Example 8: Visual Question Answering

**Pipeline task:** `visual-question-answering`  
**Pretrained model:** `dandelin/vilt-b32-finetuned-vqa`


In [ ]:
vqa = pipeline(
    task="visual-question-answering",
    model="dandelin/vilt-b32-finetuned-vqa",
    device=DEVICE
)

questions = [
    "How many birds are visible?",
    "What animals are shown?",
    "What color are the birds?"
]

for question in questions:
    answers = vqa(
        image=image2,
        question=question,
        top_k=3
    )

    print("\nQuestion:", question)

    for answer in answers:
        print(
            f"{answer['answer']}: "
            f"{answer['score']:.4f}"
        )


# Example 9: OWL-ViT Zero-Shot Object Detection

**Pipeline task:** `zero-shot-object-detection`  
**Pretrained model:** `google/owlvit-base-patch32`


In [ ]:
zero_shot_detector = pipeline(
    task="zero-shot-object-detection",
    model="google/owlvit-base-patch32",
    device=DEVICE
)

candidate_labels = [
    "bird",
    "person",
    "car",
    "tree",
    "building"
]

zero_shot_results = zero_shot_detector(
    image1,
    candidate_labels=candidate_labels,
    threshold=0.1
)

for result in zero_shot_results:
    print(result)

zero_shot_image = draw_boxes(
    image1,
    zero_shot_results,
    threshold=0.1
)

show_image(
    zero_shot_image,
    "OWL-ViT zero-shot detection"
)


# Example 10: DINOv2 Image Feature Extraction

**Pipeline task:** `image-feature-extraction`  
**Pretrained model:** `facebook/dinov2-small`


In [ ]:
feature_extractor = pipeline(
    task="image-feature-extraction",
    model="facebook/dinov2-small",
    device=DEVICE,
    pool=True
)

features_1 = feature_extractor(image1)
features_2 = feature_extractor(image2)
features_1_again = feature_extractor(image1)

vector_1 = torch.tensor(
    np.asarray(features_1)
).flatten().float()

vector_2 = torch.tensor(
    np.asarray(features_2)
).flatten().float()

vector_1_again = torch.tensor(
    np.asarray(features_1_again)
).flatten().float()

different_similarity = torch.nn.functional.cosine_similarity(
    vector_1.unsqueeze(0),
    vector_2.unsqueeze(0)
).item()

same_similarity = torch.nn.functional.cosine_similarity(
    vector_1.unsqueeze(0),
    vector_1_again.unsqueeze(0)
).item()

print("Embedding shape:", vector_1.shape)
print("Different-image similarity:", round(different_similarity, 4))
print("Same-image similarity:", round(same_similarity, 4))


# Summary

| No. | Pipeline task | Model |
|---:|---|---|
| 1 | `image-classification` | ViT |
| 2 | `image-classification` | ResNet-50 |
| 3 | `zero-shot-image-classification` | CLIP |
| 4 | `object-detection` | DETR |
| 5 | `image-segmentation` | Mask2Former |
| 6 | `depth-estimation` | Depth Anything |
| 7 | `image-to-text` | BLIP |
| 8 | `visual-question-answering` | ViLT |
| 9 | `zero-shot-object-detection` | OWL-ViT |
| 10 | `image-feature-extraction` | DINOv2 |

The `pipeline()` function automatically loads the model, processor and task-specific post-processing logic.
